# Wavesurfer-Video-DLC Alignment

Align wavesurfer acquisition data with video frames via sync signal.

Default behavior: loads existing alignment templates from
`alignment_templates/cam2/` on MICROSCOPE. Recomputes only if missing.

Outputs go to `pipeline_outputs_v2/alignment/` on MICROSCOPE.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
DATES = None

# Camera index (cam2 = DLC camera)
CAM_IDX = 2

# Overwrite mode: "skip" (default), "prompt", or "force"
OVERWRITE = "skip"

# Force recompute even if existing template is available
FORCE_RECOMPUTE = False

In [ ]:
import datetime
import json
import logging
from pathlib import Path

import numpy as np

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.alignment.timebase import load_or_compute_alignment

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
align_params = defaults["alignment"]
align_params["cam_idx"] = CAM_IDX

output_root = resolver.pipeline_outputs("alignment")
cam_name = f"cam{CAM_IDX}"

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    print(f"\n{'='*60}")
    print(f"Aligning {animal_id}: {len(dates)} dates")
    print(f"{'='*60}")

    n_loaded, n_computed, n_skipped, n_error = 0, 0, 0, 0
    for date in dates:
        date_str = date.strftime("%Y%m%d")
        out_path = output_root / cam_name / animal_id / f"{date_str}.npz"

        # Overwrite check
        if OVERWRITE == "skip" and out_path.exists():
            n_skipped += 1
            continue

        try:
            result = load_or_compute_alignment(
                animal_id, date, resolver, align_params,
                force_recompute=FORCE_RECOMPUTE,
            )

            if result.source == "loaded":
                n_loaded += 1
            else:
                n_computed += 1

            # Save to pipeline_outputs_v2/alignment/
            out_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez(
                str(out_path),
                sig_camIdx__idx_ws=result.sig_camIdx__idx_ws,
                sig_wsIdx__idx_cam=result.sig_wsIdx__idx_cam,
                slope_ws_per_camFrame=result.slope_ws_per_camFrame,
                intercept_ws_samples=result.intercept_ws_samples,
                Fs_ws_input=result.Fs_ws,
                source=result.source,
                allow_pickle=False,
            )

            # Per-session params.json
            params_path = output_root / "params" / animal_id / f"{date_str}.json"
            params_path.parent.mkdir(parents=True, exist_ok=True)
            record = {
                "session_id": f"{animal_id}_{date_str}",
                "animal": animal_id,
                "date": date.isoformat(),
                "run_timestamp": datetime.datetime.now().isoformat(),
                "skipped": False,
                "params": align_params,
                "source": result.source,
                "processing_log": [],
            }
            with open(params_path, "w") as f:
                json.dump(record, f, indent=2, default=str)

        except (FileNotFoundError, ValueError) as e:
            n_error += 1
            logging.warning("Skipping %s/%s: %s", animal_id, date_str, e)

    print(f"  Loaded: {n_loaded}, Computed: {n_computed}, Skipped: {n_skipped}, Errors: {n_error}")